# PageRank nas redes de passes — Copa do Mundo de 2022

Erick Alberti · Josué Santos · Leonardo Bicca · Luigi Porto  
Matemática Discreta — FGV EMAp

In [ ]:
# Instalação, se necessário: %pip install statsbombpy pandas numpy networkx matplotlib scipy
# Importa as bibliotecas
from statsbombpy import sb
from functools import lru_cache
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import networkx as nx

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Seleção e partidas analisadas
TIME_ALVO = "Argentina"
ids_jogos = [
    3857300,  # Arábia Saudita
    3857289,  # México
    3857264,  # Polônia
    3869151,  # Austrália
    3869321,  # Países Baixos
    3869519,  # Croácia
    3869685,  # França
]

# False: PageRank do recebedor; True: PageRank do fornecedor.
INVERTER_GRAFO = True

# False: toda a partida (inclui prorrogação); True: apenas até a primeira substituição.
RECORTE_ATE_PRIMEIRA_SUBSTITUICAO = False

# Para comparar partidas, o padrão é manter o mesmo amortecimento.
MODO_ALPHA = "fixo"             # "fixo" ou "empirico"
ALPHA_FIXO = 0.85

# Pisos/ponderações: ajustados com a grade e os eventos dos jogos selecionados.
CONSTANTE_BASE_XG = 0.05
FATOR_POSSE_XG = 0.50

NOMES_JOGOS = {
    3857300: "Arábia Saudita", 3857289: "México", 3857264: "Polônia",
    3869151: "Austrália", 3869321: "Países Baixos",
    3869519: "Croácia", 3869685: "França",
}

In [ ]:
# Matriz oficial 8x12 de Expected Threat (Karun Singh)
XT_GRID = np.array([
    [0.00638303, 0.00779616, 0.00844854, 0.00977659, 0.01126267, 0.01248344, 0.01473596, 0.01745060, 0.02122129, 0.02756312, 0.03485072, 0.03792590],
    [0.00750072, 0.00878589, 0.00942382, 0.01059490, 0.01214719, 0.01384540, 0.01611813, 0.01870347, 0.02401521, 0.02953272, 0.04066992, 0.04647721],
    [0.00887990, 0.00977745, 0.01001304, 0.01110462, 0.01269174, 0.01429128, 0.01685596, 0.01935132, 0.02412240, 0.02855202, 0.05491138, 0.06442595],
    [0.00941056, 0.01082722, 0.01016549, 0.01132376, 0.01262646, 0.01484598, 0.01689528, 0.01997070, 0.02385149, 0.03511326, 0.10805102, 0.25745362],
    [0.00941056, 0.01082722, 0.01016549, 0.01132376, 0.01262646, 0.01484598, 0.01689528, 0.01997070, 0.02385149, 0.03511326, 0.10805102, 0.25745362],
    [0.00887990, 0.00977745, 0.01001304, 0.01110462, 0.01269174, 0.01429128, 0.01685596, 0.01935132, 0.02412240, 0.02855202, 0.05491138, 0.06442595],
    [0.00750072, 0.00878589, 0.00942382, 0.01059490, 0.01214719, 0.01384540, 0.01611813, 0.01870347, 0.02401521, 0.02953272, 0.04066992, 0.04647721],
    [0.00638303, 0.00779616, 0.00844854, 0.00977659, 0.01126267, 0.01248344, 0.01473596, 0.01745060, 0.02122129, 0.02756312, 0.03485072, 0.03792590]
])

def obter_xt(x, y):
    """Converte coordenadas StatsBomb (0..120, 0..80) nos índices da grade 8x12."""
    col = np.clip((x / 120.0 * 12).astype(int), 0, 11)
    lin = np.clip((y / 80.0 * 8).astype(int), 0, 7)
    return XT_GRID[lin, col]

# Mediana dos 96 valores da grade de xT (substitui a aproximação 0,01).
CONSTANTE_BASE_XT = float(np.median(XT_GRID))
print(f"Piso de xT (mediana da grade): {CONSTANTE_BASE_XT:.8f}")

In [ ]:
@lru_cache(maxsize=64)
def carregar_eventos(match_id):
    # Evita baixar novamente o mesmo jogo quando comparamos os modelos.
    return sb.events(match_id=int(match_id))


def calcular_xg_futuro_da_posse(passes, chutes):
    # Só conta chutes posteriores ao passe, na mesma posse e no mesmo período.
    creditos = pd.Series(0.0, index=passes.index, dtype=float)
    for (periodo, posse), bloco in passes.groupby(['period', 'possession']):
        tiros = chutes.loc[
            (chutes['period'] == periodo) & (chutes['possession'] == posse)
        ].sort_values('index')
        if tiros.empty:
            continue
        indices_chutes = tiros['index'].to_numpy()
        valores_xg = tiros['shot_statsbomb_xg'].to_numpy(dtype=float)
        acumulado_futuro = np.r_[np.cumsum(valores_xg[::-1])[::-1], 0.0]
        posicoes = np.searchsorted(indices_chutes, bloco['index'].to_numpy(), side='right')
        creditos.loc[bloco.index] = acumulado_futuro[posicoes]
    return creditos

In [ ]:
def obter_eventos_titulares(match_id, time_alvo):

    eventos = carregar_eventos(match_id)
    
    # Filtra o time e ordena cronologicamente
    eventos_time = eventos[(eventos['team'] == time_alvo) & (eventos['period'] <= 4)].sort_values(
        by=['period', 'minute', 'second', 'index']
    ).copy()

    # Localiza o índice da primeira substituição
    subs = eventos_time[eventos_time['type'] == 'Substitution']
    if RECORTE_ATE_PRIMEIRA_SUBSTITUICAO and not subs.empty:
        primeira_sub_idx = subs.iloc[0]['index']
        eventos_titulares = eventos_time[eventos_time['index'] < primeira_sub_idx].copy()
    else:
        eventos_titulares = eventos_time.copy()

    return eventos_titulares

In [ ]:
def estimar_alpha_empirico(eventos_titulares):
    
    # 1. Passes que mantêm o fluxo dentro do grafo
    passes_completos = eventos_titulares[
        (eventos_titulares['type'] == 'Pass') & 
        (eventos_titulares['pass_outcome'].isna())
    ]
    n_continuacoes = len(passes_completos)

    # 2. Eventos que encerram a cadeia de passes daquela posse
    passes_errados = eventos_titulares[
        (eventos_titulares['type'] == 'Pass') & 
        (eventos_titulares['pass_outcome'].notna())
    ]
    chutes = eventos_titulares[eventos_titulares['type'] == 'Shot']
    perdas = eventos_titulares[eventos_titulares['type'].isin(['Dispossessed', 'Miscontrol'])]

    n_encerramentos = len(passes_errados) + len(chutes) + len(perdas)

    # Probabilidade empírica de a posse continuar por mais um passe
    total = n_continuacoes + n_encerramentos
    if total == 0:
        return ALPHA_FIXO
    alpha = n_continuacoes / total
    # O PageRank precisa de 0 < alpha < 1.
    alpha = float(np.clip(alpha, 0.01, 0.99))
    
    return alpha

In [ ]:
def obter_df_passes(MATCH_ID, TIME_ALVO):

    # 1. Baixa todos os eventos da partida
    eventos = carregar_eventos(MATCH_ID)

    # Filtra apenas os eventos da equipe escolhida e ordena cronologicamente
    eventos_time = eventos[(eventos['team'] == TIME_ALVO) & (eventos['period'] <= 4)].sort_values(by=['period', 'minute', 'second', 'index']).copy()

    # 2. Identifica o momento exato da PRIMEIRA SUBSTITUIÇÃO do time
    subs = eventos_time[eventos_time['type'] == 'Substitution']
    if RECORTE_ATE_PRIMEIRA_SUBSTITUICAO and not subs.empty:
        primeira_sub_idx = subs.iloc[0]['index']
        minuto_sub = subs.iloc[0]['minute']
        eventos_titulares = eventos_time[eventos_time['index'] < primeira_sub_idx].copy()
        print(f"Recorte aplicado até a 1ª substituição aos {minuto_sub} min.")
    else:
        eventos_titulares = eventos_time.copy()
        print("Nenhuma substituição encontrada; partida completa utilizada.")

    for coluna in ['pass_outcome', 'under_pressure', 'pass_shot_assist', 'pass_assisted_shot_id']:
        if coluna not in eventos_titulares.columns:
            eventos_titulares[coluna] = np.nan

    # 3. Extrai e limpa os PASSES COMPLETOS
    passes = eventos_titulares[
        (eventos_titulares['type'] == 'Pass') & 
        (eventos_titulares['pass_outcome'].isna())  # NaN = passe completo
    ].dropna(subset=['player', 'pass_recipient', 'location', 'pass_end_location']).copy()
    
    
    
    if passes.empty:
        raise ValueError(f'Sem passes válidos para {TIME_ALVO} na partida {MATCH_ID}.')

    # Desembrulha as coordenadas [x, y] (Campo StatsBomb: 120x80, gol adversário em x=120, y=40)
    passes[['x_inicio', 'y_inicio']] = pd.DataFrame(passes['location'].map(lambda p: p[:2]).tolist(), index=passes.index)
    passes[['x_fim', 'y_fim']] = pd.DataFrame(passes['pass_end_location'].map(lambda p: p[:2]).tolist(), index=passes.index)

    # 4. Engenharia de Atributos para os Pesos das Arestas (w_ij)
    # A) Ganho linear em direção ao gol (Delta X)
    passes['delta_x'] = passes['x_fim'] - passes['x_inicio'] 
    passes['avanco_positivo'] = passes['delta_x'].clip(lower=0)

    # B) Redução da distância euclidiana até o centro do gol adversário (120, 40)
    dist_inicio = np.hypot(120 - passes['x_inicio'], 40 - passes['y_inicio'])
    dist_fim = np.hypot(120 - passes['x_fim'], 40 - passes['y_fim'])
    passes['ganho_proximidade_gol'] = (dist_inicio - dist_fim).clip(lower=0)

    # C) Flags táticas (converte NaN para False/0)
    passes['sob_pressao'] = passes['under_pressure'].fillna(False).astype(int)
    passes['passe_para_chute'] = passes['pass_shot_assist'].fillna(False).astype(int)

    # D) Peso Composto Sugerido (exemplo prontamente utilizável no NetworkX):
    # Peso base 1.0 + ganho normalizado em direção ao gol + bônus por pressão (1.5x) ou assistência para chute (+5.0)
    passes['peso_tatico'] = (
        1.0 
        + (passes['ganho_proximidade_gol'] / 10.0) 
        + (passes['passe_para_chute'] * 5.0)
    ) * np.where(passes['sob_pressao'] == 1, 1.5, 1.0)

    xt_inicio = obter_xt(passes['x_inicio'], passes['y_inicio'])
    xt_fim = obter_xt(passes['x_fim'], passes['y_fim'])
    passes['delta_xt'] = xt_fim - xt_inicio
    passes['peso_xt'] = CONSTANTE_BASE_XT + passes['delta_xt'].clip(lower=0)



    # a. Extrai todos os chutes do recorte com seu ID, posse e xG
    chutes = eventos_titulares[eventos_titulares['type'] == 'Shot'][
        ['id', 'period', 'index', 'possession', 'shot_statsbomb_xg']
    ].copy()
    chutes['shot_statsbomb_xg'] = pd.to_numeric(
        chutes['shot_statsbomb_xg'], errors='coerce'
    ).fillna(0.0).clip(lower=0.0)

    # b. xA direto: liga cada passe ao chute que ele assistiu (se houver)
    mapa_xa = chutes.set_index('id')['shot_statsbomb_xg'].to_dict()
    mapa_periodo = chutes.set_index('id')['period'].to_dict()
    mapa_posse = chutes.set_index('id')['possession'].to_dict()
    mapa_indice = chutes.set_index('id')['index'].to_dict()
    if 'pass_assisted_shot_id' in passes.columns:
        passes['xa_direto'] = passes['pass_assisted_shot_id'].map(mapa_xa).fillna(0.0)
        valido = (
            passes['period'].eq(passes['pass_assisted_shot_id'].map(mapa_periodo))
            & passes['possession'].eq(passes['pass_assisted_shot_id'].map(mapa_posse))
            & (passes['index'] < passes['pass_assisted_shot_id'].map(mapa_indice))
        )
        passes.loc[~valido, 'xa_direto'] = 0.0
    else:
        passes['xa_direto'] = 0.0

    # c. xG da posse: soma o xG dos chutes que ocorreram após cada passe.
    passes['xg_posse'] = calcular_xg_futuro_da_posse(passes, chutes)

    # d. Peso final combinando piso base + assistência direta (xA) + construção da posse (xGChain)
    passes['peso_xg'] = (
        CONSTANTE_BASE_XG
        + passes['xa_direto']
        + (FATOR_POSSE_XG * passes['xg_posse'])
    )



    # 5. Dataset Principal de Arestas (Passes)
    colunas_passes = [
        'minute', 'second', 'player', 'pass_recipient',
        'x_inicio', 'y_inicio', 'x_fim', 'y_fim',
        'delta_x', 'avanco_positivo', 'ganho_proximidade_gol',
        'sob_pressao', 'passe_para_chute', 'peso_tatico',
        'delta_xt', 'xa_direto', 'xg_posse', 'peso_xt', 'peso_xg'
        ]
    df_passes = passes[colunas_passes].reset_index(drop=True)

    # 6. Dataset Auxiliar: Inícios de Posse (para o vetor de teletransporte v) e Finalizações (nó absorvedor)
    recuperacoes = eventos_titulares[eventos_titulares['type'].isin(['Ball Recovery', 'Interception'])]
    vetor_inicio_posse = recuperacoes['player'].value_counts(normalize=True).to_dict()

    finalizacoes = eventos_titulares[eventos_titulares['type'] == 'Shot'][['minute', 'second', 'player', 'shot_outcome', 'shot_statsbomb_xg']]

    print(f"Passes completos válidos: {len(df_passes)}")
    print(f"Jogadores observados no grafo: {len(set(df_passes['player']) | set(df_passes['pass_recipient']))}")
    return df_passes

In [ ]:
def calibrar_parametros_xg(lista_match_ids, time_alvo):
    # Mediana de xG dos chutes e equilíbrio entre assistência e participação na posse.
    bases = [obter_df_passes(match_id, time_alvo) for match_id in lista_match_ids]
    xg_chutes = []
    for match_id in lista_match_ids:
        eventos = obter_eventos_titulares(match_id, time_alvo)
        tiros = eventos.loc[eventos['type'] == 'Shot', 'shot_statsbomb_xg']
        xg_chutes.extend(pd.to_numeric(tiros, errors='coerce').dropna().loc[lambda s: s > 0].tolist())

    base_xg = float(np.median(xg_chutes)) if xg_chutes else 1.0
    total_direto = sum(base['xa_direto'].sum() for base in bases)
    total_posse = sum(base['xg_posse'].sum() for base in bases)
    fator_posse = float(total_direto / total_posse) if total_posse > 0 else 0.0
    return base_xg, fator_posse


CONSTANTE_BASE_XG, FATOR_POSSE_XG = calibrar_parametros_xg(ids_jogos, TIME_ALVO)
print(f"Piso xG (mediana de chutes): {CONSTANTE_BASE_XG:.6f}")
print(f"Fator do xG da posse: {FATOR_POSSE_XG:.6f}")

In [ ]:
def calcular_pagerank_futebol(MATCH_ID, TIME_ALVO, coluna_peso=None, vetor_personalizacao=None, inverter_grafo=INVERTER_GRAFO):
    
    # 1. Carrega o dataframe correspondente e calcula o alpha
    df_passes = obter_df_passes(MATCH_ID, TIME_ALVO)
    alpha = (
        estimar_alpha_empirico(obter_eventos_titulares(MATCH_ID, TIME_ALVO))
        if MODO_ALPHA == "empirico" else ALPHA_FIXO
    )
    
    # 2. Agrega os passes entre cada par (passador -> recebedor)
    if coluna_peso is None:
        arestas = df_passes.groupby(['player', 'pass_recipient']).size().reset_index(name='peso_final')
    else:
        # Garante que não existam pesos negativos
        if (df_passes[coluna_peso] < 0).any():
            raise ValueError(f"A coluna '{coluna_peso}' contém valores negativos, o que viola a Cadeia de Markov.")
        arestas = df_passes.groupby(['player', 'pass_recipient'])[coluna_peso].sum().reset_index(name='peso_final')

    # 3. Constrói o grafo direcionado ponderado
    G = nx.DiGraph()
    for _, row in arestas.iterrows():
        if inverter_grafo:
            G.add_edge(row['pass_recipient'], row['player'], weight=row['peso_final'])
        else:
            G.add_edge(row['player'], row['pass_recipient'], weight=row['peso_final'])

    # 4. Executa o PageRank (algoritmo idêntico independente da métrica de peso)
    scores = nx.pagerank(
        G,
        alpha=alpha,
        weight='weight',
        personalization=vetor_personalizacao,
        tol=1e-08
    )

    # 5. Retorna ordenado em DataFrame para facilitar comparações
    ranking = pd.DataFrame(list(scores.items()), columns=['Jogador', f'PageRank_{coluna_peso or "contagem"}'])
    return ranking.sort_values(by=ranking.columns[1], ascending=False).reset_index(drop=True)

In [ ]:
def analisar_varias_partidas_selecao(lista_match_ids, time_alvo, coluna_peso=None,
                                      inverter_grafo=INVERTER_GRAFO, retornar_partidas=False):
    """
    Recebe uma lista de match_ids de uma mesma seleção, executa a função
    calcular_pagerank_futebol para cada jogo e retorna a média de importância
    de cada jogador ao longo dessas partidas.
    """
    todos_rankings = []
    nome_coluna_pr = f'PageRank_{coluna_peso or "contagem"}'

    for match_id in lista_match_ids:
        print(f"\n--- Processando partida {match_id} ({time_alvo}) ---")
        ranking_jogo = calcular_pagerank_futebol(
            MATCH_ID=match_id,
            TIME_ALVO=time_alvo,
            coluna_peso=coluna_peso,
            inverter_grafo=inverter_grafo
        )
        ranking_jogo['match_id'] = match_id
        todos_rankings.append(ranking_jogo)

    # Junta os resultados de todos os jogos em um único DataFrame
    df_consolidado = pd.concat(todos_rankings, ignore_index=True)

    # Calcula a média de PageRank e a quantidade de partidas analisadas por jogador
    df_media = df_consolidado.groupby('Jogador', as_index=False).agg(
        Jogos_Analisados=('match_id', 'count'),
        Media_PageRank=(nome_coluna_pr, 'mean')
    )

    df_media = df_media.sort_values(by='Media_PageRank', ascending=False).reset_index(drop=True)
    return (df_media, df_consolidado) if retornar_partidas else df_media

In [ ]:
# Executa os três PageRanks nos sete jogos e calcula as médias por jogador.
modelos = {'Básico': None, 'xT': 'peso_xt', 'xG híbrido': 'peso_xg'}
medias = {}
rankings_partidas = {}

for modelo, coluna in modelos.items():
    medias[modelo], rankings_partidas[modelo] = analisar_varias_partidas_selecao(
        lista_match_ids=ids_jogos,
        time_alvo=TIME_ALVO,
        coluna_peso=coluna,
        inverter_grafo=INVERTER_GRAFO,
        retornar_partidas=True,
    )

# Reúne os resultados sem agregar as redes dos jogos.
ranking_final = None
for modelo, tabela in medias.items():
    tabela = tabela.rename(columns={
        'Media_PageRank': f'PR_{modelo}',
        'Jogos_Analisados': f'Jogos_{modelo}',
    })
    ranking_final = tabela if ranking_final is None else ranking_final.merge(
        tabela, on='Jogador', how='outer'
    )

ranking_final = ranking_final.sort_values('PR_Básico', ascending=False).reset_index(drop=True)
for modelo in modelos:
    ranking_final[f'Pos_{modelo}'] = ranking_final[f'PR_{modelo}'].rank(
        ascending=False, method='min'
    ).astype('Int64')

print(ranking_final.to_string(index=False))
print('Médias calculadas somente nos jogos em que cada jogador aparece.')

In [ ]:
# Visualização: médias de PageRank e mudanças de posição.
fig, axs = plt.subplots(1, 2, figsize=(14, 6))
selecionados = ranking_final.head(12).iloc[::-1]
y = np.arange(len(selecionados))

for i, modelo in enumerate(modelos):
    axs[0].barh(y + (i - 1)*0.26, selecionados[f'PR_{modelo}'], height=0.25, label=modelo)
axs[0].set_yticks(y, selecionados['Jogador'])
axs[0].set_title('PageRank médio nos jogos analisados')
axs[0].set_xlabel('Média do PageRank')
axs[0].legend()

mudancas = ranking_final.head(12).iloc[::-1].copy()
axs[1].barh(y - 0.17, mudancas['Pos_Básico'] - mudancas['Pos_xT'], height=0.32, label='xT')
axs[1].barh(y + 0.17, mudancas['Pos_Básico'] - mudancas['Pos_xG híbrido'], height=0.32, label='xG híbrido')
axs[1].set_yticks(y, mudancas['Jogador'])
axs[1].axvline(0, color='black', linewidth=0.8)
axs[1].set_title('Mudanças de posição em relação ao básico')
axs[1].set_xlabel('Posições ganhas (positivo)')
axs[1].legend()
plt.tight_layout()

Path('figuras').mkdir(exist_ok=True)
Path('resultados').mkdir(exist_ok=True)
plt.savefig('figuras/comparacao_medias.png', dpi=180, bbox_inches='tight')
plt.show()
ranking_final.to_csv('resultados/ranking_medio.csv', index=False)

In [ ]:
# Compara os resultados individuais dos sete confrontos.
from scipy.stats import spearmanr

comparacao_jogos = []
for match_id in ids_jogos:
    tabelas = {}
    for modelo, tabela in rankings_partidas.items():
        coluna = next(col for col in tabela.columns if col.startswith('PageRank_'))
        jogo = tabela.loc[tabela['match_id'] == match_id, ['Jogador', coluna]].copy()
        tabelas[modelo] = jogo.set_index('Jogador')[coluna].rename(modelo)

    alinhados = pd.concat(tabelas.values(), axis=1)
    # Para comparar postos, consideramos os mesmos jogadores em todos os modelos.
    posicoes = alinhados.rank(ascending=False, method='min')
    linhas = {
        'match_id': match_id, 'Adversário': NOMES_JOGOS.get(match_id, str(match_id)),
        'Líder básico': alinhados['Básico'].idxmax(),
        'Líder xT': alinhados['xT'].idxmax(),
        'Líder xG': alinhados['xG híbrido'].idxmax(),
        'Mudanças xT': int((posicoes['Básico'] != posicoes['xT']).sum()),
        'Mudanças xG': int((posicoes['Básico'] != posicoes['xG híbrido']).sum()),
        'Spearman xT': spearmanr(alinhados['Básico'], alinhados['xT']).statistic,
        'Spearman xG': spearmanr(alinhados['Básico'], alinhados['xG híbrido']).statistic,
    }
    comparacao_jogos.append(linhas)

resumo_jogos = pd.DataFrame(comparacao_jogos)
print(resumo_jogos.to_string(index=False))
resumo_jogos.to_csv('resultados/comparacao_partidas.csv', index=False)

In [ ]:
# Visualização: estabilidade dos rankings de cada partida.
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
x = np.arange(len(resumo_jogos))
axs[0].bar(x - .18, resumo_jogos['Mudanças xT'], .36, label='xT')
axs[0].bar(x + .18, resumo_jogos['Mudanças xG'], .36, label='xG híbrido')
axs[0].set_xticks(x, resumo_jogos['Adversário'], rotation=35, ha='right')
axs[0].set_ylabel('Jogadores que mudaram de posição')
axs[0].set_title('Alterações por partida')
axs[0].legend()

axs[1].plot(x, resumo_jogos['Spearman xT'], marker='o', label='Básico × xT')
axs[1].plot(x, resumo_jogos['Spearman xG'], marker='s', label='Básico × xG híbrido')
axs[1].set_xticks(x, resumo_jogos['Adversário'], rotation=35, ha='right')
axs[1].set_ylim(-0.05, 1.05)
axs[1].set_ylabel('Correlação de Spearman')
axs[1].set_title('Semelhança entre rankings')
axs[1].legend()
plt.tight_layout()
plt.savefig('figuras/comparacao_partidas.png', dpi=180, bbox_inches='tight')
plt.show()